# Missing Values

are common in statistical data. This notebook gives an introduction to how to identify/handle them.

## Load Packages and Extra Functions

In [1]:
using Printf

include("src/printmat.jl");

# NaN and missing

The `NaN` (Not-a-Number) can be used to indicate that a floating point number is missing or otherwise strange. For other types of data (for instance, integers), use `missing` instead.

Most computations involving NaN/missing give `NaN` or `missing` as a result.

Financial data is often on floating point form and `NaN` is easier than `missing` to work with (since `NaN` is part of the floating point specification, while `missing` is an add-on). This might suggest using `NaN` rather than `missing`.

In [2]:
println(2.0 + NaN," ",2 + missing)

NaN missing


# Loading Data

The next cell replaces `-999.99` by `NaN` or `missing` in the matrix `data`. This is a common scenario when `data` has been loaded from a data set (csv file, say). See the tutorial on loading and saving data for more information.

In [3]:
data = [1.0 -999.99;
        3.0 13.0]

z = replace(data,-999.99=>NaN)           #replace -999.99 by NaN or missing
printblue("z: ")
printmat(z)

z: 
     1.000       NaN
     3.000    13.000



## Testing for NaN/missing in an Array

You can test whether a number is `NaN` or `missing` by using `isunordered()`. (Use `isnan()` or `ismissing()` if you want to test specifically for one of them.) 

In [4]:
if any(isunordered,z)                  #check if any NaN/missins
  println("data has some NaN/missing")
end

data has some NaN/missing


# Disregarding NaN/missing in a Vector

can often be done by just `filter()` the vector to keep only elements that are non-NaN/missing.

In [5]:
sum(filter(!isunordered,z))    #finds all elements that are not unordered, and sums them

17.0

# Prune All Rows (of a Matrix) with any NaN/missing

It is a (fairly) common procedure in statistics to throw out all cases with NaN/missing values. For instance, let `z` be a matrix and suppose `z[t,:]` contains one or more `NaN/missing` values. It is then common (for instance, in linear regressions) to throw out that entire row of the matrix.

The function `Cases2Keep(z,dims=2)` will create a bitvector with true for all rows in `z` that has no NaN/missing.

For statistical computations, you may also consider the [NaNStatistics.jl](https://github.com/brenhinkeller/NaNStatistics.jl) package. 

In [6]:
"""
    Cases2Keep

Indicate rows (or cols) without NaN/missing
"""
Cases2Keep(z,dims=2) = .!vec(any(isunordered,z;dims));

In [7]:
#z = [0 100;1 NaN;2 21.0]               #try this too
z = [0 100;1 missing;2 21]              #try this too

vc = Cases2Keep(z)
printblue("z and vc:")
printmat(z,vc;colNames=["col 1","col 2","vc"])

z2 = view(z,vc,:)           #view of rows without NaN/missing, as an alternative do z[vc,:]
printblue("z2: view of all rows with without any NaN/missing:")
printmat(z2)

z and vc:
     col 1     col 2        vc
     0       100         1    
     1       missing     0    
     2        21         1    

z2: view of all rows with without any NaN/missing:
     0       100    
     2        21    



## Converting a View to a Standard Type (extra)\*

Once you have pruned all rows with `missing`s, you may want to convert the matrix to, for instance, Float64. This might simplify some of the later code, in particular, if you plan to send this matrix to R/Python/etc. Notice that if there were no missing (just NaN), then no conversion is needed, but you may still want to do `copy(z2)` to create an independent matrix.

As an alternative, consider `disallowmissing()` from the [`Missings.jl`](https://github.com/JuliaData/Missings.jl) package.

In [8]:
println("The type of z2 is\n", typeof(z2))

z3 = convert.(Float64,z2)                       #z3 will be Float64 and a also an independent copy
z4 = copy(z2)
println("\nThe types of z3 and z4 are\n",typeof(z3),"\n",typeof(z4))     

The type of z2 is
SubArray{Union{Missing, Int64}, 2, Matrix{Union{Missing, Int64}}, Tuple{Vector{Int64}, Base.Slice{Base.OneTo{Int64}}}, false}

The types of z3 and z4 are
Matrix{Float64}
Matrix{Union{Missing, Int64}}
